In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

## Задача 1. 

Поиграемся с датасетом про диабет (он совсем легкий). 

In [31]:
df = pd.read_csv('Diabetes Binary Classification.csv')
df.head()

,Number of times pregnant,Plasma glucose concentration a 2 hours in an oral glucose tolerance test,Diastolic blood pressure (mm Hg),Triceps skin fold thickness (mm),2-Hour serum insulin (mu U/ml),Body mass index (weight in kg/(height in m)^2),Diabetes pedigree function,Age (years),Class variable (0 or 1)
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


Целевая переменная тут явно обозначена как Class variable (очевидно, есть у человека диабет или нет)

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                                                                    Non-Null Count  Dtype  
---  ------                                                                    --------------  -----  
 0   Number of times pregnant                                                  768 non-null    int64  
 1   Plasma glucose concentration a 2 hours in an oral glucose tolerance test  768 non-null    int64  
 2   Diastolic blood pressure (mm Hg)                                          768 non-null    int64  
 3   Triceps skin fold thickness (mm)                                          768 non-null    int64  
 4   2-Hour serum insulin (mu U/ml)                                            768 non-null    int64  
 5   Body mass index (weight in kg/(height in m)^2)                            768 non-null    float64
 6   Diabetes pedigree function                                         

In [33]:
df.describe()

,Number of times pregnant,Plasma glucose concentration a 2 hours in an oral glucose tolerance test,Diastolic blood pressure (mm Hg),Triceps skin fold thickness (mm),2-Hour serum insulin (mu U/ml),Body mass index (weight in kg/(height in m)^2),Diabetes pedigree function,Age (years),Class variable (0 or 1)
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [34]:
df.isnull().sum()

Number of times pregnant                                                    0
Plasma glucose concentration a 2 hours in an oral glucose tolerance test    0
Diastolic blood pressure (mm Hg)                                            0
Triceps skin fold thickness (mm)                                            0
2-Hour serum insulin (mu U/ml)                                              0
Body mass index (weight in kg/(height in m)^2)                              0
Diabetes pedigree function                                                  0
Age (years)                                                                 0
Class variable (0 or 1)                                                     0
dtype: int64

In [35]:
df['Class variable (0 or 1)'].value_counts()

Class variable (0 or 1)
0    500
1    268
Name: count, dtype: int64

In [36]:
X = df.drop('Class variable (0 or 1)', axis=1)
y = df['Class variable (0 or 1)']

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),  
    ('model', LogisticRegression(class_weight='balanced', solver='liblinear'))  
])

params = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__penalty': ['l1', 'l2']
}

grid_pima = GridSearchCV(pipe, params, scoring='f1', cv=5)
grid_pima.fit(X_train, y_train)

best_model = grid_pima.best_estimator_
y_pred = best_model.predict(X_test)
print("лучшие параметры:", grid_pima.best_params_)
print(classification_report(y_test, y_pred))

лучшие параметры: {'model__C': 1, 'model__penalty': 'l2'}
              precision    recall  f1-score   support

           0       0.84      0.79      0.82       150
           1       0.65      0.72      0.68        81

    accuracy                           0.77       231
   macro avg       0.74      0.75      0.75       231
weighted avg       0.77      0.77      0.77       231



In [38]:
pipe_svc = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='linear', class_weight='balanced'))
])

params_svc = {
    'model__C': [0.01, 0.1, 1, 10]
}

grid_svc = GridSearchCV(pipe_svc, params_svc, scoring='f1_macro', cv=5)
grid_svc.fit(X_train, y_train)

print("лучшие параметры:", grid_svc.best_params_)

y_test_pred = grid_svc.predict(X_test)
print(classification_report(y_test, y_test_pred))

лучшие параметры: {'model__C': 0.1}
              precision    recall  f1-score   support

           0       0.83      0.81      0.82       150
           1       0.67      0.69      0.68        81

    accuracy                           0.77       231
   macro avg       0.75      0.75      0.75       231
weighted avg       0.77      0.77      0.77       231



обе модели показали сопоставимую точность — 77% и f1 — 68%. без class_weight='balanced' и гридсерча было хуже

## Задача 2. 

Второй датасет - про покупателей велосипедов. 

In [39]:
data = pd.read_csv('bike_buyers_clean.csv')
data.head()

,ID,Marital Status,Gender,Income,Children,Education,Occupation,Home Owner,Cars,Commute Distance,Region,Age,Purchased Bike
0,12496,Married,Female,40000,1,Bachelors,Skilled Manual,Yes,0,0-1 Miles,Europe,42,No
1,24107,Married,Male,30000,3,Partial College,Clerical,Yes,1,0-1 Miles,Europe,43,No
2,14177,Married,Male,80000,5,Partial College,Professional,No,2,2-5 Miles,Europe,60,No
3,24381,Single,Male,70000,0,Bachelors,Professional,Yes,1,5-10 Miles,Pacific,41,Yes
4,25597,Single,Male,30000,0,Bachelors,Clerical,No,0,0-1 Miles,Europe,36,Yes


Пытаемся по характеристикам человека понять, купит он велик или нет. 

In [40]:
X = data.drop(columns=['Purchased Bike'])
y = data['Purchased Bike']

X = pd.get_dummies(X, drop_first=True)

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [42]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', solver='liblinear'))
])

params_lr = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__penalty': ['l1', 'l2']
}

grid_lr = GridSearchCV(pipe_lr, params_lr, cv=5, scoring='f1_macro', n_jobs=-1)
grid_lr.fit(X_train, y_train)

print("лучшие параметры:", grid_lr.best_params_)

y_pred_lr = grid_lr.predict(X_test)
print(classification_report(y_test, y_pred_lr))

лучшие параметры: {'model__C': 1, 'model__penalty': 'l1'}
              precision    recall  f1-score   support

          No       0.65      0.65      0.65       156
         Yes       0.62      0.62      0.62       144

    accuracy                           0.64       300
   macro avg       0.64      0.64      0.64       300
weighted avg       0.64      0.64      0.64       300



In [43]:
pipe_svc = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='linear', class_weight='balanced'))
])

params_svc = {
    'model__C': [0.01, 0.1, 1, 10, 100]
}

grid_svc = GridSearchCV(pipe_svc, params_svc, cv=5, scoring='f1_macro', n_jobs=-1)
grid_svc.fit(X_train, y_train)

print("лучшие параметры:", grid_svc.best_params_)

y_pred_svc = grid_svc.predict(X_test)
print(classification_report(y_test, y_pred_svc))

лучшие параметры: {'model__C': 10}
              precision    recall  f1-score   support

          No       0.66      0.63      0.64       156
         Yes       0.62      0.64      0.63       144

    accuracy                           0.64       300
   macro avg       0.64      0.64      0.64       300
weighted avg       0.64      0.64      0.64       300



лучше работает с class_weight='balanced'. без гридсерча f1 был ниже.. в целом получилось средненько и с lr, и с svc: метрики невысокие, особенно precision и recall на тесте, но выровненные 